# Strategy - Momentum

In [ ]:
# ============================================================
# S01 - Cross-Sectional Momentum
# ============================================================

# ============================================================
# Imports
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

# Load Quantiacs Crypto Daily dataset

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Extract required fields
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    # --------------------------------------------------------
    # Calculate daily returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # Volatility estimate
    # --------------------------------------------------------

    # Used later for risk adjustment

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Momentum Signals
    # --------------------------------------------------------

    # 21-day momentum
    mom21 = close / close.shift(time=21) - 1

    # 63-day momentum
    mom63 = close / close.shift(time=63) - 1

    # --------------------------------------------------------
    # Risk Adjusted Momentum
    # --------------------------------------------------------

    risk_adj_mom = (
        mom63 /
        (vol20 + 1e-6)
    )

    # --------------------------------------------------------
    # Drawdown Quality
    # --------------------------------------------------------

    # Assets near highs are preferred

    rolling_high = close.rolling(
        time=90
    ).max()

    drawdown_factor = (
        close /
        (rolling_high + 1e-6)
    )

    # --------------------------------------------------------
    # Composite Momentum Score
    # --------------------------------------------------------

    score = (
        0.45 * mom21.rank("asset").fillna(0)
        +
        0.35 * risk_adj_mom.rank("asset").fillna(0)
        +
        0.20 * drawdown_factor.rank("asset").fillna(0)
    )

    # Restrict to liquid universe

    score = score * is_liquid

    # --------------------------------------------------------
    # Select Strongest Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize Portfolio
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    return weights.fillna(0)

# ============================================================
# Generate Portfolio Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean Submission Weights
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Write Submission File
# ============================================================

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field            equity  relative_return  volatility  underwater  \
time                                                               
2026-06-05  1350.071642        -0.054293    0.776676   -0.608912   
2026-06-06  1339.327964        -0.007958    0.776581   -0.612024   
2026-06-07  1388.582847         0.036776    0.776551   -0.597756   
2026-06-08  1388.856534         0.000197    0.776450   -0.597676   
2026-06-09  1361.742104        -0.019523    0.776378   -0.605531   

field       max_drawdown  sharpe_ratio  mean_return  bias  instruments  \
time                                                                     
2026-06-05     -0.922149      1.284981     0.998014   1.0         37.0   
2026-06-06     -0.922149      1.283819     0.996990   1.0         37.0   
2026-06-07     -0.922149      1.288068     1.000251   1.0         37.0   
2026-06-08     -0.922149      1.287923     1.000007   1.0         37.0   
2026-06-09     -0.922149      1.285296     0.997876   1.0         37.0   

field       avg_turnover  avg_holding_time  
time                                        
2026-06-05      0.110287         19.458112  
2026-06-06      0.110284         19.457360  
2026-06-07      0.110269         19.456924  
2026-06-08      0.110255         19.457083  
2026-06-09      0.110242         19.470502  

Final Metrics:
field
equity              1361.742104
sharpe_ratio           1.285296
max_drawdown          -0.922149
avg_turnover           0.110242
avg_holding_time      19.470502
Name: 2026-06-09 00:00:00, dtype: float64